# Agent的高级用法-ToolStrategy

## 1、ToolStrategy的多种结构化输出方式：schema参数

### 1.1 Pydantic类型


使用CloseAI平台

In [7]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
# 从.env文件中加载环境变量
load_dotenv(override=True)
CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")
model_with_closeai = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

使用OpenRouter平台

In [2]:
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
import os

load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")
model_with_openrouter = ChatOpenRouter(
    model="openai/gpt-5.4-mini",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

使用阿里云百炼平台

In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

举例1：

In [9]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo)
)

response = agent.invoke({
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
        ]
    }
)

# for msg in response["messages"]:
#     msg.pretty_print()
rprint(response)
# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='dae67b64-43b4-4227-aa30-f4720e2a23a2'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 61,
                    'prompt_tokens': 349,
                    'total_tokens': 410,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 349}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-9182fcdc-9109-9ebf-89e5-3ac13b8d541e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f65f3-c74d-7a12-a286-813f3e89be05-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_83395234b4db4add94666fae',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 349,
                'output_tokens': 61,
                'total_tokens': 410,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='songhk@atguigu.com' phone='12345678912'",
            name='ContactInfo',
            id='75c8b4d3-b327-4af7-8a19-f943f06975a2',
            tool_call_id='call_83395234b4db4add94666fae'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

举例2：添加工具的调用

In [16]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal, TypedDict, Annotated
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"

# 定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")

# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                "请分析指定客户的情况："
                                "1. 先搜索客户数据库了解最新情况 "
                                "2. 如果是VIP客户，则发送感谢邮件 "
                                "3. 基于搜索结果生成结构化分析报告 "
                                "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)
# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

{
    'messages': [
        HumanMessage(
            content='今天天气如何',
            additional_kwargs={},
            response_metadata={},
            id='8ea7ed77-b168-49bb-9268-cecff9968440'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 24,
                    'prompt_tokens': 620,
                    'total_tokens': 644,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 620}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-10541c54-c5b9-92f6-872f-aed047bd5c61',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6607-0b96-7073-b45f-c7af7edbc6df-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '今天天气如何'},
                    'id': 'call_2271c5ec7d204c0c91d3971d',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 620,
                'output_tokens': 24,
                'total_tokens': 644,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='关于客户今天天气如何，无记录',
            name='search_customer_database',
            id='cd873271-eb06-4a04-a000-5132801b9e20',
            tool_call_id='call_2271c5ec7d204c0c91d3971d'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 11,
                    'prompt_tokens': 673,
                    'total_tokens': 684,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 673}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-c67f3598-3a96-9e66-bed8-cff5f9752778',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6607-107f-7490-a4a3-6882e3676573-0',
            tool_calls=[
                {
                    'name': 'CustomerAnalysis',
                    'args': {},
                    'id': 'call_25a3013b46cd459ca8e67e8c',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 673,
                'output_tokens': 11,
                'total_tokens': 684,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: customer_name=None customer_tier='潜在客户' 
recent_activity=None spending_level=None send_email=False",
            name='CustomerAnalysis',
            id='5a6e1ec6-ed48-4d44-8202-8cf4da10924a',
            tool_call_id='call_25a3013b46cd459ca8e67e8c'
        )
    ],
    'structured_response': CustomerAnalysis(
        customer_name=None,
        customer_tier='潜在客户',
        recent_activity=None,
        spending_level=None,
        send_email=False
    )
}

### 1.2 TypedDict

举例1：

In [18]:
from typing import TypedDict, Annotated
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(TypedDict):
    """用户的联系方式"""
    name: Annotated[str, ..., "用户姓名"]
    email: Annotated[str, ..., "用户邮箱地址"]
    phone: Annotated[str, ..., "用户的手机号"]

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo)
)

response = agent.invoke({
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
        ]
    }
)

# for msg in response["messages"]:
#     msg.pretty_print()
rprint(response)
# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='adbcbfbb-fb45-40f1-8f1e-e76082717bcc'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 61,
                    'prompt_tokens': 327,
                    'total_tokens': 388,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 327}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-fab685a3-a74b-9ffc-87ca-64ed6612159f',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6612-1c9d-73b1-9c0c-ce26bc8303b6-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_9258032220bb4c56ba2f8935',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 327,
                'output_tokens': 61,
                'total_tokens': 388,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': 
'12345678912'}",
            name='ContactInfo',
            id='07bff46b-bec9-45be-8b41-9110f8beeaa7',
            tool_call_id='call_9258032220bb4c56ba2f8935'
        )
    ],
    'structured_response': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'}
}

举例2：添加工具的调用

In [4]:
from langchain_core.messages import SystemMessage
from typing import Literal, Optional
from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"

# 使用 TypedDict 定义客户分析报告 Schema
class CustomerAnalysis(TypedDict):
    """客户分析报告"""
    customer_name: Annotated[Optional[str], None, "客户姓名"]
    customer_tier: Annotated[Literal["潜在客户", "普通客户", "VIP客户", "流失风险"], "潜在客户", "客户等级"]
    recent_activity: Annotated[Optional[str], None, "最近活动"]
    spending_level: Annotated[Optional[Literal["低", "中", "高"]], None, "消费水平"]
    send_email: Annotated[bool, False, "是否已发送感谢邮件"]

# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                "请分析指定客户的情况："
                                "1. 先搜索客户数据库了解最新情况 "
                                "2. 如果是VIP客户，则发送感谢邮件 "
                                "3. 基于搜索结果生成结构化分析报告 "
                                "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint("result:", result)
# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

result:
{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='d2790c02-076c-48f2-904f-0b75da04da15'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 22,
                    'prompt_tokens': 606,
                    'total_tokens': 628,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 606}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-edf5bb2c-9643-9cc4-87bc-1ab4260f5af6',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6b66-823e-7800-b160-fc779bccc54f-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_3568512d573e4e61a79a2da7',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 606,
                'output_tokens': 22,
                'total_tokens': 628,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='b1435476-56dd-4f39-b764-ae4684440740',
            tool_call_id='call_3568512d573e4e61a79a2da7'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 23,
                    'prompt_tokens': 682,
                    'total_tokens': 705,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 682}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-65fb02bf-ce32-92b7-9bfa-386cac633470',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6b66-876f-7672-9591-118864b85de3-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '张三'},
                    'id': 'call_efe820f98f2642eea7f5ec5f',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 682,
                'output_tokens': 23,
                'total_tokens': 705,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='已向 张三 发送感谢邮件',
            name='send_email',
            id='aa49f6cc-9314-4dde-a931-be40038d4c3d',
            tool_call_id='call_efe820f98f2642eea7f5ec5f'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 88,
                    'prompt_tokens': 733,
                    'total_tokens': 821,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 733}
                },
                'model_provider': 'openai',


### 1.3 JsonSchema类型

JSON Schema是提供一个标准的 JSON Schema 字典来定义结构。适合需要与多种编程语言交互或进行复杂数据约束定义的场景。

举例1：


In [5]:
from pydantic_core.core_schema import json_schema
from typing import TypedDict, Annotated
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

json_schema = {
    "title": "ContactInfo",
    "description": "用户的联系方式",
    "type": "object",
    "properties": {
        "name": {
            "description": "用户姓名",
            "type": "string"
        },
        "email": {
            "description": "用户邮箱地址",
            "type": "string"
        },
        "phone": {
            "description": "用户的手机号",
            "type": "string"
        }
    },
    "required": [
        "name",
        "email",
        "phone"
    ]
}

agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=json_schema),
)

response = agent.invoke({
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
        ]
    }
)

# for msg in response["messages"]:
#     msg.pretty_print()
rprint(response)
# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='1aa1335c-8878-4583-a77c-6810a2600fda'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 61,
                    'prompt_tokens': 349,
                    'total_tokens': 410,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 349}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-ceacd5ba-c001-9efb-a736-9646de156beb',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6b6e-a688-7041-9f85-134a629a881a-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_1ae4252840bd44758a7d1c8e',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 349,
                'output_tokens': 61,
                'total_tokens': 410,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': 
'12345678912'}",
            name='ContactInfo',
            id='e8c9e894-9952-478a-a359-2528adc0cd4b',
            tool_call_id='call_1ae4252840bd44758a7d1c8e'
        )
    ],
    'structured_response': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'}
}

举例2：

In [7]:
from langchain_core.messages import SystemMessage
from typing import Literal, Optional
from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"

# 使用 json_schema 定义客户分析报告 Schema
customer_analysis_schema = {
    "title": "CustomerAnalysis",
    "type": "object",
    "description": "客户分析报告",
    "properties": {
        "customer_name": {
            "type": "string",
            "default": "",
            "description": "客户姓名"
        },
        "customer_tier": {
            "type": "string",
            "enum": ["潜在客户", "普通客户", "VIP客户", "流失风险"],
            "default": "潜在客户",
            "description": "客户等级"
        },
        "recent_activity": {
            "type": "string",
            "default": "",
            "description": "最近活动"
        },
        "spending_level": {
            "type": "string",
            "enum": ["低", "中", "高"],
            "default": "低",
            "description": "消费水平"
        },
        "send_email": {
            "type": "boolean",
            "default": False,
            "description": "是否已发送感谢邮件"
        }
    },
    # 所有字段都是必须输出的
    "required": ["customer_name", "customer_tier", "recent_activity", "spending_level"]
}


# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                "请分析指定客户的情况："
                                "1. 先搜索客户数据库了解最新情况 "
                                "2. 如果是VIP客户，则发送感谢邮件 "
                                "3. 基于搜索结果生成结构化分析报告 "
                                "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=customer_analysis_schema)
)

# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint("result:", result)
# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

result:
{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='4e563427-705e-4549-88bf-b0393a36dd4f'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 22,
                    'prompt_tokens': 632,
                    'total_tokens': 654,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 632}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-95e27a99-0406-95ac-b5c9-4f32135d5d43',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6b71-f393-77c1-8042-6bbc85800249-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_b416ada59a2e46cfb363dc57',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 632,
                'output_tokens': 22,
                'total_tokens': 654,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='a4c15e08-36ac-4a6f-ad93-b82a39de993f',
            tool_call_id='call_b416ada59a2e46cfb363dc57'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 23,
                    'prompt_tokens': 708,
                    'total_tokens': 731,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 708}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-fe095ace-36a0-9cfc-8667-fd82295544ee',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6b71-f743-73c0-bcb9-f30d9ad225a3-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '张三'},
                    'id': 'call_dda797b5867e449daed7016c',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 708,
                'output_tokens': 23,
                'total_tokens': 731,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='已向 张三 发送感谢邮件',
            name='send_email',
            id='6357b0c4-d773-4a04-a919-25521ee27003',
            tool_call_id='call_dda797b5867e449daed7016c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 103,
                    'prompt_tokens': 759,
                    'total_tokens': 862,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 759}
                },
                'model_provider': 'openai',

### 1.4 @dataclass类型

@dataclass是Python 3.7引入的一个装饰器，用于简化数据存储类的定义。


举例1：

In [10]:
from dataclasses import dataclass
from  pydantic.fields import Field
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

@dataclass
class ContactInfo:
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo)
)
response = agent.invoke(
 {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
        ]
 }
)
# for msg in response["messages"]:
#     msg.pretty_print()
rprint(response["structured_response"])

ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')

举例2：调用工具

In [12]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from dataclasses import dataclass
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"
@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"

# 使用Dataclass定义Schema
@dataclass
class CustomerAnalysis:
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户", description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")
# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                "请分析指定客户的情况："
                                "1. 先搜索客户数据库了解最新情况 "
                                "2. 如果是VIP客户，则发送感谢邮件 "
                                "3. 基于搜索结果生成结构化分析报告 "
                                "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)
# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})
# 处理结果
rprint("result:", result)
# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

result:
{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='cbff266b-c680-4ac5-8b84-e21e987e0205'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 22,
                    'prompt_tokens': 609,
                    'total_tokens': 631,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 609}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-b6889d89-8446-99c9-a533-45a1a1790fdb',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6b76-131c-7d31-9a74-408d1a3cdd12-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_67a1cc5d9b484bb9af28227d',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 609,
                'output_tokens': 22,
                'total_tokens': 631,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='82159189-82fd-49b3-9150-8e2e6480022a',
            tool_call_id='call_67a1cc5d9b484bb9af28227d'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 23,
                    'prompt_tokens': 685,
                    'total_tokens': 708,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 685}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-1bd44fa9-07a5-97d1-a64d-89b38b36a3b0',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6b76-16a0-79b0-96b4-87e826813bc0-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '张三'},
                    'id': 'call_67358461a3114ff0bd95cecf',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 685,
                'output_tokens': 23,
                'total_tokens': 708,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='已向 张三 发送感谢邮件',
            name='send_email',
            id='f9002bcd-e37b-4fe8-833b-39c0ae9ab475',
            tool_call_id='call_67358461a3114ff0bd95cecf'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 103,
                    'prompt_tokens': 736,
                    'total_tokens': 839,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 736}
                },
                'model_provider': 'openai',

### 1.5 多schema联合模式

ToolStrategy允许指定多个类型“ Union[类型1, 类型2] ”这种写法，LLM能够根据输入文本的内容，智能地选择最合适的一个数据模型（Schema）来生成结构化输出，但是最终会只有一种类型输出。

适用于根据不同输入内容，生成不同的结构化输出的场景，但是底层工具转换结构化输出只会转换成一种结构化类型输出。

举例：

In [13]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema=Union[ContactInfo, EventInfo]
    )
)

response = agent.invoke(
 {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
        ]
 }
)

for msg in response["messages"]:
    msg.pretty_print()

rprint(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_c01a13d27f5149c88f6eedc4)
 Call ID: call_c01a13d27f5149c88f6eedc4
  Args:
    email: shkstart@atguigu.com
    name: 小明
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'


ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')

In [14]:
response = agent.invoke(
 {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：2026年高考报名人数突破1200万")
        ]
 }
)

for msg in response["messages"]:
    msg.pretty_print()

rprint(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：2026年高考报名人数突破1200万
================================== Ai Message ==================================
Tool Calls:
  EventInfo (call_6624eb5fe0e84d248574bb79)
 Call ID: call_6624eb5fe0e84d248574bb79
  Args:
    date: 2026年
    event_name: 高考报名人数突破1200万
================================= Tool Message =================================
Name: EventInfo

Returning structured response: event_name='高考报名人数突破1200万' date='2026年'


EventInfo(event_name='高考报名人数突破1200万', date='2026年')

## 2、自定义工具消息：tool_message_content参数

我们可以通过ToolStrategy的 tool_message_content 参数定制其消息内容，将指定的内容写入对话历史的提示信息，这样做的好处如下：

1. 在最终用户可见的对话流中，使用 更自然的消息 替代原始数据

2. 用简短的确认信息替代可能很长的数据块， 减少token消耗

举例：

In [15]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")
agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema=ContactInfo,
        tool_message_content="已成功提取信息"
    ),
)

response = agent.invoke({
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
        ]
    }
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='298afe25-2245-4520-b229-c9e3559ad139'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 61,
                    'prompt_tokens': 349,
                    'total_tokens': 410,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 349}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-118316f2-3512-9ce9-b708-3a1b37dcaf57',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6bac-cdf2-7770-a7d8-b2a2982412bc-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_b9e52ad47f34464cb04a418e',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 349,
                'output_tokens': 61,
                'total_tokens': 410,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='已成功提取信息',
            name='ContactInfo',
            id='3b505317-3a49-4195-a173-9041275ccb44',
            tool_call_id='call_b9e52ad47f34464cb04a418e'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}